# Bài 6: Hàm (Function)

Hàm là cách **đóng gói một đoạn code thành khối tái sử dụng**. Trong AI, bạn sẽ viết hàm liên tục: `normalize(data)`, `accuracy(y_true, y_pred)`, `train_one_epoch(...)`, `load_config(path)`... Code gọn, dễ test, dễ đọc hơn nhiều so với copy-paste.

Mục tiêu bài học:
- Định nghĩa hàm bằng `def`, tham số, `return`
- Tham số mặc định & gọi hàm bằng keyword argument
- Trả về nhiều giá trị (thực chất là tuple + unpacking)
- Phạm vi biến (scope): biến cục bộ vs biến toàn cục
- `*args` và `**kwargs`
- Docstring — viết tài liệu cho hàm

> Nhắc lại cách học: đọc lý thuyết → chạy thử ví dụ → tự làm bài tập, đừng xem lời giải trước.

## 1. Định nghĩa và gọi hàm

Cú pháp: `def tên_hàm(tham_số):` rồi thân hàm **thụt lề**. `return` trả kết quả về cho nơi gọi. Hàm không có `return` sẽ trả về `None`.

In [1]:
def mean(values):
    return sum(values) / len(values)

accuracies = [0.72, 0.91, 0.85]
print(mean(accuracies))
print(mean([1, 2, 3, 4]))   # dùng lại với dữ liệu khác

def say_hello(name):
    print(f"Xin chào {name}")   # không có return

result = say_hello("An")
print(result)   # None

0.8266666666666667
2.5
Xin chào An
None


## 2. Tham số mặc định & keyword argument

- **Tham số mặc định**: `def f(x, y=10)` — nếu không truyền `y` thì dùng 10.
- **Keyword argument**: gọi `f(x=1, y=2)` — truyền theo tên, không cần nhớ thứ tự. Cực kỳ phổ biến trong thư viện ML (`LogisticRegression(C=1.0, max_iter=100)`).

In [2]:
def make_config(lr=0.001, batch_size=32, optimizer="adam"):
    return {"lr": lr, "batch_size": batch_size, "optimizer": optimizer}

print(make_config())                                 # dùng toàn bộ mặc định
print(make_config(lr=0.01))                          # chỉ đổi lr
print(make_config(optimizer="sgd", batch_size=64))   # thứ tự tuỳ ý khi dùng keyword

{'lr': 0.001, 'batch_size': 32, 'optimizer': 'adam'}
{'lr': 0.01, 'batch_size': 32, 'optimizer': 'adam'}
{'lr': 0.001, 'batch_size': 64, 'optimizer': 'sgd'}


## 3. Trả về nhiều giá trị

Python cho phép `return a, b` — thực chất trả về **một tuple**, rồi bạn unpacking (bài 4) ở nơi gọi.

In [3]:
def min_max(values):
    return min(values), max(values)

lo, hi = min_max([0.82, 0.65, 0.41, 0.30])
print(f"min={lo}, max={hi}")

print(min_max([3, 1, 2]))   # là một tuple

min=0.3, max=0.82
(1, 3)


## 4. Phạm vi biến (scope)

- Biến tạo **trong hàm** là biến **cục bộ** — chỉ tồn tại bên trong hàm đó.
- Biến tạo **ngoài hàm** là biến **toàn cục** — hàm đọc được, nhưng **gán** vào nó sẽ tạo ra một biến cục bộ mới (không đổi biến toàn cục).
- Thói quen tốt: hàm nhận dữ liệu qua **tham số** và trả kết quả qua **return**, tránh sửa biến toàn cục.

In [4]:
lr = 0.001   # biến toàn cục

def show_lr():
    print("Đọc biến toàn cục:", lr)

def try_change_lr():
    lr = 0.5   # tạo biến CỤC BỘ mới, không đụng tới lr toàn cục
    print("Trong hàm:", lr)

show_lr()
try_change_lr()
print("Ngoài hàm:", lr)   # vẫn 0.001

def double(x):
    result = x * 2
    return result

print(double(4))
# print(result)   # bỏ comment để thấy lỗi NameError: result chỉ tồn tại trong hàm

Đọc biến toàn cục: 0.001
Trong hàm: 0.5
Ngoài hàm: 0.001
8


### Lưu ý: list/dict là mutable nên hàm có thể sửa "ngầm" dữ liệu gốc

Nhắc lại bài 4: list là mutable. Khi truyền list vào hàm, hàm nhận **cùng một list** (không phải bản sao), nên `append` bên trong sẽ ảnh hưởng list bên ngoài.

In [5]:
def add_epoch_loss(losses, value):
    losses.append(value)   # sửa trực tiếp list gốc

history = [0.8, 0.6]
add_epoch_loss(history, 0.4)
print(history)   # [0.8, 0.6, 0.4] -- đã bị thay đổi

# Muốn giữ list gốc: làm việc trên bản sao
def add_epoch_loss_safe(losses, value):
    new_losses = losses.copy()
    new_losses.append(value)
    return new_losses

h2 = add_epoch_loss_safe(history, 0.3)
print(history, h2)

[0.8, 0.6, 0.4]
[0.8, 0.6, 0.4] [0.8, 0.6, 0.4, 0.3]


## 5. `*args` và `**kwargs`

- `*args`: gom **mọi tham số vị trí thừa** thành một **tuple**.
- `**kwargs`: gom **mọi keyword argument thừa** thành một **dict**.

Bạn sẽ gặp chúng khắp nơi trong code framework (PyTorch, scikit-learn) khi hàm cần nhận số lượng tham số tuỳ ý hoặc chuyển tiếp tham số cho hàm khác.

In [6]:
def total(*args):
    print("args =", args, type(args))
    return sum(args)

print(total(1, 2, 3))
print(total(10, 20))

def show_options(**kwargs):
    print("kwargs =", kwargs, type(kwargs))
    for key, value in kwargs.items():
        print(f"{key} = {value}")

show_options(lr=0.01, epochs=5, optimizer="sgd")

args = (1, 2, 3) <class 'tuple'>
6
args = (10, 20) <class 'tuple'>
30
kwargs = {'lr': 0.01, 'epochs': 5, 'optimizer': 'sgd'} <class 'dict'>
lr = 0.01
epochs = 5
optimizer = sgd


### Dùng `*` và `**` khi gọi hàm (unpacking)

Ngược lại, khi **gọi** hàm, `*list` bung list thành các tham số vị trí, `**dict` bung dict thành keyword argument. Rất hay dùng để truyền cả config dict vào hàm.

In [7]:
def make_config(lr=0.001, batch_size=32, optimizer="adam"):
    return {"lr": lr, "batch_size": batch_size, "optimizer": optimizer}

params = {"lr": 0.01, "batch_size": 128}
print(make_config(**params))   # tương đương make_config(lr=0.01, batch_size=128)

nums = [1, 2, 3]
print(total(*nums))            # tương đương total(1, 2, 3)

{'lr': 0.01, 'batch_size': 128, 'optimizer': 'adam'}
args = (1, 2, 3) <class 'tuple'>
6


## 6. Docstring — viết tài liệu cho hàm

Chuỗi `"""..."""` ngay dòng đầu tiên trong thân hàm gọi là **docstring**, mô tả hàm làm gì, nhận gì, trả gì. `help(hàm)` và IDE đều hiển thị nó. Viết docstring là thói quen của AI Engineer chuyên nghiệp.

In [8]:
def accuracy(y_true, y_pred):
    """Tính độ chính xác (accuracy).

    Args:
        y_true: list nhãn thật.
        y_pred: list nhãn dự đoán, cùng độ dài với y_true.

    Returns:
        float trong [0, 1]: tỉ lệ dự đoán đúng.
    """
    correct = 0
    for t, p in zip(y_true, y_pred):   # zip: ghép từng cặp phần tử của 2 list
        if t == p:
            correct += 1
    return correct / len(y_true)

print(accuracy([1, 0, 1, 1], [1, 0, 0, 1]))   # 0.75
help(accuracy)

0.75
Help on function accuracy in module __main__:

accuracy(y_true, y_pred)
    Tính độ chính xác (accuracy).

    Args:
        y_true: list nhãn thật.
        y_pred: list nhãn dự đoán, cùng độ dài với y_true.

    Returns:
        float trong [0, 1]: tỉ lệ dự đoán đúng.



## Bài tập thực hành

1. Viết hàm `square(x)` trả về bình phương của `x`. In `square(5)` và `square(1.5)`.
2. Viết hàm `normalize(values)` nhận list số, trả về list mới đã chuẩn hoá **min-max**: mỗi phần tử `(v - min) / (max - min)`. Thử với `[10, 20, 30, 50]` (kết quả mong đợi: `[0.0, 0.25, 0.5, 1.0]`).
3. Viết hàm `describe(values)` trả về **3 giá trị**: min, max, trung bình. Gọi hàm và unpack thành `lo, hi, avg`, rồi in ra.
4. Viết hàm `build_model_name(base, size="base", version=1)` trả về chuỗi dạng `"resnet-base-v1"`. Gọi 3 cách: chỉ truyền `base`; truyền `base` và `size="large"`; truyền `base`, `version=3` bằng keyword.
5. Viết hàm `average(*args)` nhận số lượng số tuỳ ý và trả về trung bình. In `average(1, 2, 3)` và `average(10, 20, 30, 40)`.
6. **(Nâng cao)** Viết hàm `count_labels(labels)` trả về dict đếm số lần xuất hiện mỗi nhãn (ôn lại bài 5). Sau đó viết hàm `most_common(labels)` **gọi lại** `count_labels` và trả về `(nhãn, số lần)` xuất hiện nhiều nhất. Thử với `["cat", "dog", "cat", "bird", "dog", "cat"]` → `("cat", 3)`.

In [9]:
# Bài 1: square

In [10]:
# Bài 2: normalize

In [11]:
# Bài 3: describe + unpack

In [12]:
# Bài 4: build_model_name

In [13]:
# Bài 5: average với *args

In [14]:
# Bài 6 (nâng cao): count_labels và most_common

## Gợi ý

- Bài 2: tính `lo = min(values)`, `hi = max(values)` một lần ngoài vòng lặp, rồi tạo list mới bằng `for` + `append`.
- Bài 4: dùng f-string: `f"{base}-{size}-v{version}"`.
- Bài 5: `args` là tuple, dùng `sum(args) / len(args)`.
- Bài 6: trong `most_common`, duyệt `counts.items()` và giữ lại cặp có số lần lớn nhất (hoặc dùng `max(counts, key=counts.get)`).

## Lời giải mẫu

⚠️ Chỉ chạy ô dưới đây **sau khi** bạn đã tự làm xong 6 bài tập ở trên.

In [15]:
# --- Lời giải mẫu ---

# Bài 1
def square(x):
    return x * x

print(square(5), square(1.5))

# Bài 2
def normalize(values):
    lo = min(values)
    hi = max(values)
    result = []
    for v in values:
        result.append((v - lo) / (hi - lo))
    return result

print(normalize([10, 20, 30, 50]))

# Bài 3
def describe(values):
    return min(values), max(values), sum(values) / len(values)

lo, hi, avg = describe([3, 7, 5, 1])
print(f"min={lo}, max={hi}, avg={avg}")

# Bài 4
def build_model_name(base, size="base", version=1):
    return f"{base}-{size}-v{version}"

print(build_model_name("resnet"))
print(build_model_name("resnet", size="large"))
print(build_model_name("resnet", version=3))

# Bài 5
def average(*args):
    return sum(args) / len(args)

print(average(1, 2, 3))
print(average(10, 20, 30, 40))

# Bài 6
def count_labels(labels):
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return counts

def most_common(labels):
    counts = count_labels(labels)
    best_label = None
    best_count = 0
    for label, count in counts.items():
        if count > best_count:
            best_label, best_count = label, count
    return best_label, best_count

print(most_common(["cat", "dog", "cat", "bird", "dog", "cat"]))

25 2.25
[0.0, 0.25, 0.5, 1.0]
min=1, max=7, avg=4.0
resnet-base-v1
resnet-large-v1
resnet-base-v3
2.0
25.0
('cat', 3)


## Tổng kết

Bạn đã học được:
- ✅ Định nghĩa hàm với `def`, tham số, `return` (không có `return` → `None`)
- ✅ Tham số mặc định và keyword argument
- ✅ Trả về nhiều giá trị (tuple) + unpacking
- ✅ Phạm vi biến: cục bộ vs toàn cục; hàm có thể sửa ngầm list/dict được truyền vào
- ✅ `*args` (tuple), `**kwargs` (dict) và `*`/`**` khi gọi hàm
- ✅ Viết docstring cho hàm

**Bài tiếp theo:** Bài 7 — Comprehension, generator, lambda: cách viết code ngắn gọn kiểu Python (`[x*2 for x in data]`), rất hay dùng khi xử lý dữ liệu. Làm xong bài tập ở trên rồi báo mình nhé!